# ProteinLens-OS: Interactive Training, Ablation & 3D-Pocket DMS Notebook (Google Colab / Local)

**Institution**: IIIT Nagpur — Computational Biology & AI Lab  
**Repository**: `https://github.com/Arjun-E-Naik/ProtienLens`  

This notebook demonstrates the complete reproducible training and evaluation pipeline of **ProteinLens-OS** in under 2 minutes on Google Colab (GPU T4 or CPU):
1. Clone the open-source repository and verify model assets.
2. Extract live **ESM-2** (`facebook/esm2_t6_8M_UR50D`) evolutionary embeddings and Masked-Marginal Log-Likelihood Ratios (LLR).
3. Train our **Multi-Label PyTorch MLP** (`320 -> 512 -> 256 -> 500`, `BCEWithLogitsLoss` + `AdamW`) on a strict leakage-free 80/20 CAFA 6 split.
4. Audit **Fixed-Point True Path Rule DAG Propagation** proving `0.00%` post-repair monotonicity violations.
5. Benchmark **Multi-Modal Dual-Axis Clinical Risk Fusion** on verified `TP53`, `HBB`, and `INS` variants.

In [ ]:
# Step 1: Clone ProteinLens-OS from GitHub & Install Dependencies
!git clone https://github.com/Arjun-E-Naik/ProtienLens.git
%cd ProtienLens
!pip install -q torch transformers scikit-learn scipy matplotlib

In [ ]:
# Step 2: Run Real Multi-Label PyTorch MLP Training & Learning Curve Generation (~15s on CPU, ~3s on Colab T4)
!python training/train_live_demo.py

In [ ]:
# Step 3: Visualize the Generated Learning Curve
from IPython.display import Image, display
display(Image('benchmarks/output/fig3_training_curve.png'))

In [ ]:
# Step 4: Live ESM-2 Embedding Extraction + 3D-Pocket Zero-Shot Deep Mutational Scanning (TP53 p.R273H)
import sys
sys.path.insert(0, 'backend')
from pathlib import Path
from ml.engine import ProteinLensScientificEngine

engine = ProteinLensScientificEngine(Path('backend/assets'))
p53_seq = [p['sequence'] for p in engine.presets if p['uniprot_id'] == 'P04637'][0]

res = engine.analyze_variant_and_pocket_dms(
    sequence=p53_seq, pos_1based=273, mut_aa='H', uniprot_id='P04637', run_pocket_dms=True
)
da = res['dual_axis_assessment']
print(f"Variant              : TP53 p.{res['mutation']}")
print(f"ESM2 Masked LLR      : {da['evolutionary_llr']:.4f}")
print(f"AlphaFold pLDDT      : {res['structural_pocket']['target_plddt']}")
print(f"5.0 A Pocket Contacts: {res['structural_pocket']['all_5A_contact_count']}")
print(f"Dual-Axis Risk Score : {da['composite_risk_score']} / 100 ({da['risk_tier']})")
print(f"Pocket DMS Rows      : {len(res['pocket_dms']['rows'])} sites x 20 canonical AAs")

In [ ]:
# Step 5: Run Full Publishable CAFA 6 Ablation + Clinical Missense ROC Benchmark
!python benchmarks/run_paper_benchmarks.py
display(Image('benchmarks/output/fig1_cafa6_ablation.png'))
display(Image('benchmarks/output/fig2_clinical_roc.png'))